# CS103 ScienceWorld Tutorial

이 노트북은 CS103 ScienceWorld 과제를 시작하기 위한 학생용 튜토리얼입니다.

- Part 1 = HW5 (Prompting)
- Part 2 = HW6 (RAG / Tool Use)

목표는 다음과 같습니다.

1. ScienceWorld environment를 직접 열어 보기
2. valid action space가 어떻게 생겼는지 확인하기
3. LangChain / LangGraph 기반 agent를 어떻게 연결할지 감 잡기
4. HW5와 HW6를 각각 어떤 방식으로 접근해야 하는지 이해하기


## 0. 설치 전 준비

ScienceWorld는 Python뿐 아니라 Java도 필요합니다.

- Python 3.8+
- Java 8+
- Jupyter Notebook 또는 JupyterLab

LangChain / LangGraph를 이용해 agent를 구현할 예정이므로 관련 패키지도 같이 설치합니다.


In [ ]:
# Option A: 배포된 패키지를 설치하는 경우
%pip install cs103-scienceworld langchain langgraph langchain-openai

# Option B: 저장소를 직접 clone해서 examples/ 폴더에서 실행하는 경우
# %pip install -e ..
# %pip install langchain langgraph langchain-openai


In [ ]:
from typing import Annotated, List, TypedDict

from cs103_scienceworld import (
    CS103ScienceWorldHW5Env,
    CS103ScienceWorldHW6Env,
    CS103ScienceWorldSandBoxEnv,
)

from cs103_scienceworld.assignments import (
    Assignment1PromptingTemplateAgent,
    build_assignment_2_retriever,
    create_assignment_1_env,
    create_assignment_2_env,
    get_assignment_2_recipe_documents,
)

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# 실제 모델을 연결하고 싶다면 아래 예시를 참고하세요.
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


---

## Part 1. HW5: Prompting

HW5의 핵심은 **현재 observation과 valid action list를 바탕으로 다음 행동을 고르는 prompt를 설계하는 것**입니다.

이 과제에서는 template가 제공됩니다. 여러분은 template의 TODO 부분을 채우면서 agent를 완성하면 됩니다.

특히 신경 써야 할 점은 다음과 같습니다.

- task description에서 필요한 정보를 정확히 읽기
- valid action 안에서만 행동을 고르도록 만들기
- 모델 출력이 이상할 때를 대비해 fallback을 두기
- 여러 variation에서 반복 실행해 보기


In [ ]:
hw5_env = CS103ScienceWorldHW5Env()
print("HW5 task names:", hw5_env.task_names)
hw5_env.close()


### 1.1 HW5 environment 열어 보기

먼저 task description, observation, valid actions가 어떤 식으로 주어지는지 직접 보는 것이 좋습니다.


In [ ]:
env = create_assignment_1_env(variation_idx=2)
observation, info = env.reset()

print("Task Description:")
print(info["taskDesc"])
print()
print("Initial Observation:")
print(observation)
print()
print("Some Valid Actions:")
print(env.get_valid_action_object_combinations()[:20])

env.close()


### 1.2 제공되는 HW5 template 구조

HW5에서는 `Assignment1PromptingTemplateAgent`를 상속해서 구현하게 됩니다.

보통 아래 메서드들이 핵심입니다.

- `reset()`: episode 시작 시 state 초기화
- `build_prompt()`: observation과 action list를 prompt로 정리
- `choose_action()`: 모델 또는 규칙 기반으로 action 하나 선택
- `act()`: 최종적으로 environment에 전달할 action 반환


In [ ]:
class MyHW5Agent(Assignment1PromptingTemplateAgent):
    def choose_action(self, valid_actions, prompt, observation, info):
        # TODO(HW5):
        # 1. LangChain model을 호출하거나
        # 2. LangGraph로 만든 controller를 호출한 뒤
        # 3. valid_actions 안의 action 하나를 반환하세요.
        
        # 가장 단순한 fallback 예시
        return valid_actions[0]


# 예시 실행
# from cs103_scienceworld.assignments import run_assignment_1_episode
# result = run_assignment_1_episode(MyHW5Agent(), variation_idx=0, verbose=True)
# result


### 1.3 LangGraph로 HW5를 푸는 최소 구조 예시

HW5는 아래처럼 작은 LangGraph state machine으로 풀면 자연스럽습니다.

권장 node 분해:

1. prompt 생성
2. 모델 호출
3. action 정규화
4. fallback 처리


In [ ]:
class HW5State(TypedDict):
    observation: str
    task_desc: str
    valid_actions: List[str]
    prompt: str
    chosen_action: str
    messages: Annotated[list, add_messages]


def build_hw5_graph(template_agent: Assignment1PromptingTemplateAgent):
    graph = StateGraph(HW5State)

    def make_prompt_node(state: HW5State):
        prompt = template_agent.build_prompt(
            state["observation"],
            {"taskDesc": state["task_desc"]},
            state["valid_actions"],
        )
        return {"prompt": prompt}

    def pick_action_node(state: HW5State):
        # 실제 과제에서는 여기서 LangChain model.invoke(...)를 쓸 수 있습니다.
        # 아래는 단순 fallback 예시입니다.
        return {"chosen_action": state["valid_actions"][0]}

    graph.add_node("make_prompt", make_prompt_node)
    graph.add_node("pick_action", pick_action_node)
    graph.add_edge(START, "make_prompt")
    graph.add_edge("make_prompt", "pick_action")
    graph.add_edge("pick_action", END)
    return graph.compile()


### 1.4 HW5에서 체크할 것

- prompt에 action space를 넣었는가?
- 모델 출력이 invalid할 때도 안전하게 동작하는가?
- variation을 바꿔도 구조가 유지되는가?
- 규칙 기반 fallback이 너무 강해서 모델을 사실상 안 쓰는 구조는 아닌가?


---

## Part 2. HW6: RAG / Tool Use

HW6의 핵심은 **외부 knowledge source를 조회하고, 그 결과를 environment action과 연결하는 것**입니다.

이번 과제에서는 HW5처럼 완성된 template를 제공하기보다, 여러분이 튜토리얼을 참고해서 LangGraph 기반 agent를 직접 짜는 방향이 더 자연스럽습니다.

즉 이 파트에서는 아래 요소들을 직접 조합하게 됩니다.

- recipe retriever
- ingredient parsing
- environment action chooser
- LangGraph state update


In [ ]:
hw6_env = CS103ScienceWorldHW6Env()
print("HW6 task names:", hw6_env.task_names)
hw6_env.close()

recipe_documents = get_assignment_2_recipe_documents()
print("Recipe keys:", list(recipe_documents.keys()))
print("Example recipe:")
print(recipe_documents["fruit salad"])


### 2.1 Sandbox environment

과제 env 말고도, action space나 helper를 시험해 보고 싶을 때는 sandbox env를 쓸 수 있습니다.

sandbox env에는 CS103 assignment task가 아닌 일반 task들이 들어 있습니다.


In [ ]:
sandbox_env = CS103ScienceWorldSandBoxEnv()
print("Sandbox task count:", len(sandbox_env.task_names))
print("Some sandbox tasks:", sandbox_env.task_names[:10])
sandbox_env.close()


### 2.2 외부 recipe retriever 예시

기본 제공 retriever는 단순 lexical retriever입니다. 여러분은 이 부분을 vector DB나 embedding retrieval로 교체해도 됩니다.


In [ ]:
retriever = build_assignment_2_retriever()
print(retriever.query("How do I make fruit salad?"))
print(retriever.get("fruit salad"))


### 2.3 HW6 environment 열어 보기

먼저 task description, observation, valid action space를 직접 확인해 봅시다.


In [ ]:
env = create_assignment_2_env(variation_idx=0)
observation, info = env.reset()

print("Task Description:")
print(info["taskDesc"])
print()
print("Initial Observation:")
print(observation)
print()
print("Some Valid Actions:")
print(env.get_valid_action_object_combinations()[:25])

env.close()


### 2.4 HW6를 위한 최소 LangGraph 구조 예시

HW6는 보통 아래 node들로 나누면 구현이 편합니다.

1. task parsing
2. recipe retrieval
3. ingredient extraction
4. next action choice
5. state update


In [ ]:
class HW6State(TypedDict):
    task_desc: str
    observation: str
    result_name: str
    retrieved_recipe: str
    ingredients: List[str]
    valid_actions: List[str]
    chosen_action: str
    messages: Annotated[list, add_messages]


def parse_result_name(task_desc: str) -> str:
    marker = "focus on the "
    if marker not in task_desc.lower():
        return ""
    start = task_desc.lower().index(marker) + len(marker)
    return task_desc[start:].split(".")[0].strip()


def build_hw6_graph(retriever):
    graph = StateGraph(HW6State)

    def retrieve_recipe_node(state: HW6State):
        recipe_id = retriever.query(state["result_name"], top_k=1)[0]
        return {"retrieved_recipe": retriever.get(recipe_id)}

    def extract_ingredients_node(state: HW6State):
        text = state["retrieved_recipe"]
        rhs = text.split("mix", 1)[1].strip().rstrip(".")
        ingredients = [item.strip() for item in rhs.split(",") if item.strip()]
        return {"ingredients": ingredients}

    def choose_action_node(state: HW6State):
        # 실제 과제에서는 여기서 LangChain model 또는 tool routing을 연결하면 됩니다.
        if state["valid_actions"]:
            return {"chosen_action": state["valid_actions"][0]}
        return {"chosen_action": "look around"}

    graph.add_node("retrieve_recipe", retrieve_recipe_node)
    graph.add_node("extract_ingredients", extract_ingredients_node)
    graph.add_node("choose_action", choose_action_node)
    graph.add_edge(START, "retrieve_recipe")
    graph.add_edge("retrieve_recipe", "extract_ingredients")
    graph.add_edge("extract_ingredients", "choose_action")
    graph.add_edge("choose_action", END)
    return graph.compile()


### 2.5 HW6에서 바로 출발할 수 있는 최소 예시 코드

아래 코드는 완성 답안이 아니라, 여러분이 자신의 LangGraph agent를 만들 때 출발점으로 쓸 수 있는 최소 예시입니다.


In [ ]:
retriever = build_assignment_2_retriever()
graph = build_hw6_graph(retriever)

env = create_assignment_2_env(variation_idx=1)
observation, info = env.reset()

state = {
    "task_desc": str(info["taskDesc"]),
    "observation": observation,
    "result_name": parse_result_name(str(info["taskDesc"])),
    "retrieved_recipe": "",
    "ingredients": [],
    "valid_actions": list(env.get_valid_action_object_combinations()),
    "chosen_action": "",
    "messages": [],
}

out = graph.invoke(state)
print("Result name:", out["result_name"])
print("Retrieved recipe:", out["retrieved_recipe"])
print("Ingredients:", out["ingredients"])
print("Chosen action:", out["chosen_action"])

env.close()


## 마무리

### HW5에서 기억할 것

- template의 TODO를 채워서 agent를 완성한다
- valid action 바깥의 행동을 출력하지 않도록 주의한다
- LangGraph를 쓴다면 prompt 생성, 모델 호출, action 검증을 분리하는 것이 좋다

### HW6에서 기억할 것

- 외부 recipe source를 조회하는 흐름이 중요하다
- retrieval 결과를 state에 저장하고 다음 행동에 반영해야 한다
- LangGraph node를 `retrieve -> parse -> act -> update`처럼 나누면 구현이 쉬워진다

### 자주 발생하는 문제

- Java가 설치되지 않아서 env가 안 열리는 경우
- 로컬 소스를 수정한 뒤 JAR rebuild를 하지 않은 경우
- valid action이 아닌 문자열을 반환해서 step이 꼬이는 경우
